# exp-039 — MetaQA end-to-end, direct-KG compile-execute (Colab)

Runs **exp-039** on a Colab GPU. Each arm decomposes the 750 MetaQA questions of the exp-039
evaluation set (250 per hop, drawn from the official vanilla test split), compiles every
sub-question into a KG op, executes the chain on `kb.txt` and scores the answer set against gold.
This is the **direct-KG backend, not GRAG** (ADR 0045). Arms (all 4-bit, greedy, seed 42):

| arm | model | prompt |
|---|---|---|
| `oracle` | none (CPU): gold-like decompositions | the measured EM ceiling |
| `qwen_prompted` | Qwen3.5-9B | unguided, few-shot from the MetaQA pool |
| `mistral_prompted` | Mistral-7B-Instruct-v0.3 | same |
| `mistral_ft` | Mistral-7B + exp-026 MuSiQue pool adapter | `--no-few-shot` |
| `qwen_ft` | Qwen3.5-9B + exp-036 MuSiQue pool adapter | `--no-few-shot`; **skipped while the adapter does not exist** |

Primary comparisons (pre-declared, `configs/metaqa_arm_report.json`): `mistral_ft` vs `mistral_prompted`
and `qwen_ft` vs `qwen_prompted`, per hop, exact McNemar on answer EM, Holm m = 6.

**Before you run**
1. Runtime → Change runtime type → **L4 GPU** (A100 also works).
2. Colab secret `HF_TOKEN` (notebook access on): a token whose account accepted the Mistral-7B-Instruct-v0.3 licence.
3. `thesis-backup.tar.gz` at `MyDrive/thesis-backup.tar.gz`.
4. Adapters: the exp-036 adapter is found on Drive under `MyDrive/thesis-exp-036/runs/exp-036/train/*/adapter`
   (written by the exp-036 notebook). The exp-026 adapter is restored from the archive
   (`runs/exp-026/20260920_024535/adapter`) if it is in it; otherwise set `ADAPTER_MISTRAL_DRIVE` below to a
   folder on Drive that holds it. An arm whose adapter is missing is skipped and recorded, never faked.
5. The repo is public: the clone needs no token.

**Time** (estimate, not measured on Colab): 750 rows per arm at the 3090's MuSiQue decode rate
(2.76 s/row Qwen3.5-9B exp-027, 4.12 s/row Mistral-7B exp-020, both 4-bit) is ~35–52 min per arm, so
~2.5–3.5 h for the four GPU arms. MetaQA prompts are shorter than MuSiQue's; L4 speed is unmeasured.

**If the runtime dies:** start a new runtime and Run all. `runs/exp-039` is a Drive folder, so every
finished row (`results.partial.jsonl`) is already there; the notebook re-checks out the commit it first
launched at (`launch_sha.txt` on Drive) and each decode resumes (`run_decomposer.py --resume`).

In [ ]:
# ---- settings (edit here) ----
REPO_URL = "https://github.com/AhmadiJahid/Thesis---QAv2.git"   # public: no token
BRANCH = "main"
COMMIT = ""            # "" = pin the branch head at the first launch (recorded on Drive); or a full 40-char SHA
ARCHIVE_ON_DRIVE = "/content/drive/MyDrive/thesis-backup.tar.gz"
ARCHIVE_SHA256 = "33c6072f0e4e46ab94f40f804463c512f439823b80c3e0e9dec0ee1023717d94"
KB_SHA256 = "9b4d3b3066266075ad1577b4beed68e6e112e76e1a29ed2cb00a2896d2b792af"   # metaqa/kb.txt (ADR 0045)
DRIVE_OUT = "/content/drive/MyDrive/thesis-exp-039"
EXP036_DRIVE = "/content/drive/MyDrive/thesis-exp-036"     # where the exp-036 notebook wrote its adapter
ADAPTER_MISTRAL_DRIVE = ""                                 # only if exp-026's adapter is not in the archive
WORK = "/content/work"                                     # clone -> WORK/Thesis---QAv2, data -> WORK/thesis-qav2-data
MIN_VRAM_GIB = 12                                          # 3090 peaks: 4.8 GiB Mistral 4-bit, 7.8 GiB Qwen 4-bit

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
pathlib.Path(f"{DRIVE_OUT}/runs/exp-039").mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{DRIVE_OUT}/logs").mkdir(parents=True, exist_ok=True)
assert os.path.exists(ARCHIVE_ON_DRIVE), f"archive not found: {ARCHIVE_ON_DRIVE}"
print("Drive OK; outputs ->", DRIVE_OUT)

In [ ]:
# ---- GPU gate ----
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
assert q.returncode == 0, "no GPU: Runtime -> Change runtime type -> L4 GPU"
name, mem_mib, driver = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
print(f"GPU {name}, {int(mem_mib)/1024:.1f} GiB, driver {driver}")
assert int(mem_mib) / 1024 >= MIN_VRAM_GIB, f"need >= {MIN_VRAM_GIB} GiB VRAM"
GPU_INFO = {"name": name, "memory_total_mib": int(mem_mib), "driver": driver}

In [ ]:
# ---- public clone; pin the commit on first launch, re-check it out on every re-run ----
import os, subprocess
os.makedirs(WORK, exist_ok=True)
REPO = f"{WORK}/Thesis---QAv2"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sha_file = f"{DRIVE_OUT}/launch_sha.txt"
want = open(sha_file).read().strip() if os.path.exists(sha_file) else COMMIT.strip()
if want:
    subprocess.run(["git", "fetch", "origin", want], check=False)
    subprocess.run(["git", "checkout", "--detach", want], check=True)
SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
if not os.path.exists(sha_file):
    open(sha_file, "w").write(SHA + "\n")
assert not subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"],
                          capture_output=True, text=True).stdout.strip(), "clone is not clean"
assert os.path.exists("experiments/exp-039/run.sh"), "this commit has no exp-039 (is the PR merged?)"
print("repo at", REPO, "HEAD", SHA)

In [ ]:
# ---- environment: the repo's pinned requirements ----
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
assert torch.cuda.is_available()

In [ ]:
# ---- Hugging Face login (Mistral-7B-Instruct-v0.3 is gated) ----
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("HF user:", whoami()["name"])

In [ ]:
# ---- data: MetaQA part of the archive (kb.txt, legacy questions for the masking corpus) ----
import glob, hashlib, shutil, time
local_archive = "/content/thesis-backup.tar.gz"
if not os.path.exists(local_archive):
    t = time.time(); shutil.copy(ARCHIVE_ON_DRIVE, local_archive); print(f"copied archive in {time.time()-t:.0f}s")
subprocess.run(["python", "scripts/colab_setup_data.py", "--archive", local_archive, "--sha256", ARCHIVE_SHA256,
                "--parts", "metaqa"], check=True)
kb = f"{WORK}/thesis-qav2-data/metaqa/kb.txt"
assert hashlib.sha256(open(kb, "rb").read()).hexdigest() == KB_SHA256, "kb.txt sha256 differs from ADR 0045's"
# exp-026 adapter: best effort from the archive (the setup script exits non-zero if it is absent).
r = subprocess.run(["python", "scripts/colab_setup_data.py", "--archive", local_archive, "--parts",
                    "--runs", "exp-026/20260920_024535/adapter"], capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
# runs/exp-039 IS the Drive folder: every finished row lands on Drive at once.
link = "runs/exp-039"
if not os.path.islink(link):
    assert not os.path.exists(link), f"{link} exists and is not the Drive link"
    os.symlink(f"{DRIVE_OUT}/runs/exp-039", link)
print(link, "->", os.readlink(link))

def resolve_adapters():
    found = {}
    m = "runs/exp-026/20260920_024535/adapter"
    if os.path.exists(f"{m}/adapter_config.json"):
        found["ADAPTER_MISTRAL"] = os.path.abspath(m)
    elif ADAPTER_MISTRAL_DRIVE and os.path.exists(f"{ADAPTER_MISTRAL_DRIVE}/adapter_config.json"):
        found["ADAPTER_MISTRAL"] = ADAPTER_MISTRAL_DRIVE
    q = sorted(os.path.dirname(p) for p in glob.glob(f"{EXP036_DRIVE}/runs/exp-036/train/*/adapter/adapter_model.safetensors"))
    if len(q) == 1:
        found["ADAPTER_QWEN"] = q[0]
    elif len(q) > 1:
        raise SystemExit(f"more than one exp-036 adapter on Drive, pick one explicitly: {q}")
    return found

ADAPTERS = resolve_adapters()
print("adapters:", ADAPTERS or "none found", "(an arm without its adapter is skipped and recorded)")

In [ ]:
# ---- step runner: streams the log to Drive, samples VRAM, records each step ----
import json, threading, datetime

def run_step(step):
    log_path = f"{DRIVE_OUT}/logs/{step}.log"
    peak = {"mib": 0}
    stop = threading.Event()
    def sampler():
        while not stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True).stdout.strip()
            if out:
                peak["mib"] = max(peak["mib"], int(out.splitlines()[0]))
            stop.wait(10)
    th = threading.Thread(target=sampler, daemon=True); th.start()
    started = time.time()
    env = {**os.environ, **ADAPTERS}
    with open(log_path, "a", buffering=1) as log:
        log.write(f"=== {step} start {datetime.datetime.now().isoformat()} sha {SHA} gpu {GPU_INFO}\n")
        p = subprocess.Popen(["bash", "experiments/exp-039/run.sh", step], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in p.stdout:
            log.write(line)
            if "Processed" not in line:
                print(line, end="")
        rc = p.wait()
        wall = time.time() - started
        log.write(f"=== {step} rc={rc} wall_s={wall:.0f} peak_vram_mib_sampled={peak['mib']}\n")
    stop.set(); th.join()
    record = {"step": step, "rc": rc, "wall_seconds": round(wall), "peak_vram_mib_sampled_whole_gpu": peak["mib"],
              "gpu": GPU_INFO, "git_sha": SHA, "adapters": ADAPTERS, "finished": datetime.datetime.now().isoformat()}
    with open("runs/exp-039/colab_steps.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    print(record)
    assert rc == 0, f"{step} failed (rc={rc}); see {log_path}"

In [ ]:
run_step("eval_set")   # downloads 3 text files (~6 MB, pinned mirror revision), draws the set, checks the committed manifest
run_step("smoke")      # 6 Qwen rows through decode + compile-execute: the MetaQA prompting path's first real run in v2
run_step("oracle")     # CPU: gold-like decompositions -> the EM ceiling

In [ ]:
run_step("qwen_prompted")      # resumes an interrupted decode; a finished one is skipped

In [ ]:
run_step("mistral_prompted")

In [ ]:
run_step("mistral_ft")         # skipped (and recorded) if the exp-026 adapter was not found

In [ ]:
run_step("qwen_ft")            # skipped (and recorded) until the exp-036 adapter exists on Drive

In [ ]:
# ---- report: per-arm metrics with CIs, pre-declared comparisons, Holm m=6 ----
run_step("report")
m = json.load(open("runs/exp-039/report/arm_report_metrics.json"))
for arm, cells in m["per_arm"].items():
    for hop, c in cells.items():
        print(f"{arm:17s} hop {hop}: compile {c['compiled']['mean']:.3f} execute {c['executed']['mean']:.3f} "
              f"EM {c['em']['mean']:.3f} {c['em']['ci95']} Jaccard {c['jaccard']['mean']:.3f} "
              f"any-hit {c['any_hit']['mean']:.3f} RHits@1 {c['rhits1']['mean']:.3f}")
for c in m["comparisons"]:
    if "skipped" in c:
        print("comparison", c["a"], "vs", c["b"], "SKIPPED:", c["skipped"]); continue
    print(f"{c['a']} vs {c['b']} hop {c['hop']}: EM {c['em']['difference']:+.3f} "
          f"({c['em']['only_a']}:{c['em']['only_b']}) p={c['em']['mcnemar_exact_p']:.3g} Holm p={c['em']['holm_p']:.3g}")

In [ ]:
# ---- what to copy back to the laptop ----
print(f"Everything is in {DRIVE_OUT}/ on Google Drive. Copy back to the laptop:")
print(f"  {DRIVE_OUT}/runs/exp-039/  -> C:/Users/suuser/thesis/Thesis---QAv2/runs/exp-039/  (gitignored; whole folder)")
print(f"  {DRIVE_OUT}/logs/*.log      -> runs/exp-039/logs/  (wall clock, VRAM)")
print(f"  {DRIVE_OUT}/launch_sha.txt  -> the commit for the log row ({SHA})")
print("Then: copy runs/exp-039/report/arm_report_metrics.json to experiments/exp-039/metrics.json, each arm's")
print("decomposer config.json into experiments/exp-039/, write experiments/exp-039/notes.md, complete the log row.")